Here I will add all of necesary for me things idk probably will go to gitignore

Najpopularniejsze przedrostki 
feat: (feature) – nowa funkcja lub nowa część logiki (np. feat: add SoD risk detection logic).

docs: (documentation) – zmiany wyłącznie w dokumentacji, opisach, sekcjach Markdown lub komentarzach (np. docs: update domain standards in notebook).

fix: (bug fix) – naprawienie błędu w kodzie (np. fix: correct user ID mapping in dataframe).

refactor: – zmiana struktury kodu bez zmiany jego działania (np. czyszczenie kodu, zmiana nazw zmiennych).

style: – formatowanie kodu, wcięcia, spacje (brak zmian w logice).

test: – dodanie lub poprawa testów jednostkowych.

slowka do commitow 
implement

### SAP systems applications and products; oni produkują oprogramowanie. SYSTEMS APPLICATIONS AND PRODUCTS in data processing.; zarządzanie zasobami calych przedsiebiorstw; to ogolnie jest dziedzina cyberbezpieczenstwa i oni sie zamuja ochrona tych konkretnych systemow przed atakami 

## Documentation 

pojecia systemowe
acces rule maintanance from SAP GRC ACCES CONTROL (ARA); mapowanie pojec sap grc na nasz kod 
https://help.sap.com/docs/SAP_ACCESS_CONTROL/fd442c67791b45ce8339954033f8bca0/6589e44f7d794c59e10000000a445394.html?locale=en-US
we use this rule to menage three key actions 
    functions - t-codes; (AKCJE), grouping specifing actions or transations (t-codes) performed by a user 
    acces risks - defined risks representing scenarios where a single user is assigned a combination of two mutually exclusive actions (czyli kiedy uzytkownik moze zfobic dwie akcje ktore sie wykluczaja)
    rule sets - df_sod - a pre-defined collection or repository of these audit rules (in our project, this corresponds to our df_sod matrix).
Based on this we can test access of given users. 





zasady audytowe - kody tranzakcji w ramce danych oraz poziomy ryzka w macierzy wykluczen 

ISACA ERP Audit Framework & SoD Guidelines: mapowanie poziomów ryzyka i transakcji
https://www.scribd.com/document/416388704/SAP-ERP-Audit-Assurance-Program#page=37

We use this framework to validate transaction codes (T-Codes) and map audit risk severity in our detection model:
* **Procure-to-Pay (P2P) Scope:** Focuses on standard transactions like F110/PAY01 (Payments), MIRO (Invoice Processing), and FK05/XK05 (Vendor Management). - konkretne kody transakcji
* **Severity Levels (Risk_Level):** Defines conflict levels (e.g., HIGH, CRITICAL) based on the Four-Eyes Principle—separating payment execution from invoice approval or vendor master data edits. - poziomy ryzyka w macierzy wykluczen
* **Audit Rationale:** Replicates automated user authorization testing specified in ISACA P2P Audit Guidelines (Section 4: Processing Disbursements).
Uzasadnienie audytowe: Kod powiela i automatyzuje testy uprawnień użytkownika, które są oficjalnie wymagane przez wytyczne audytowe ISACA dla procesu Procure-to-Pay.

idea 4 oczu, podział obowiazkow na cztery filary 
ISACA Journal - Four-Eyes Principle Implementation
* **Source:** [ISACA Journal (Volume 5): A Step-by-Step SoD Implementation Guide](https://www.isaca.org/resources/isaca-journal)

Internal Control Architecture & Four-Eyes Principle
     Establishes the 4-pillar internal control framework (Custody, Authorization, Recording, Execution) to isolate conflicting business roles.
    Justifies the exact pair-matching logic in our Pandas detection algorithm, ensuring that no single user holds both Authorization and Execution privileges in P2P transactions.

code notes 
sod rules to zmienna, ktora jest lista w pythonie (uporzadkowany zbior w pythonie, kazdy element ma swoja stala pozycje )
{} - w srodku sa slowniki, kazdy z nich reprezentuje regule wykluczajaca sie czyli konflikt sod 
pola wewnatrz slownika
    risk id - unikalny identyfikator reguly 
    tcode a - pierwsza kodowana transakcja w sap np fk01 - tworzenie dostawcy 
    tcdoe b - druga transakcja ktora wyklucza si e z pierwsza 
    risk level - poziom krytycznosci konfliktu; jak pilna jest rekacja na dane naruszenie 
    description - krotki opis tlumaczacy na czym polega dany konflikt 

konwersja na strukture tabelaryczna 
    pd datafrane przeksztalca liste slownikow w dwuwymiarowa tabele, dzieki czemu klucze ze wslownikow staja sie nazwami kolumn a poszczegolne slowniki staja sie wierszami 

my wybralismy te trzy konkretne konfliktu poniewaz bazuja  na standardach bezpieczenstwa, czyli dla glownego procesu finansowo zakupowego w firmach czyli procure to pay 
te trzy reguly obejmuja pelna sciezke operacyjna w firmie - od zakupu towaru przez ksiegowanie faktur i zarzadzenia odstawacami az po sama wyplate pieniedzy plus demonstruje rozne poziomy ryszyk a

u nas tylko pay01 jest uproszczone, bo normalnie to sie bardziej precyzuje 
FK01 + PAY01:

FK01 to standardowa transakcja SAP FI do tworzenia dostawcy (Create Vendor).

PAY01 użyliśmy jako uproszczonego skrótu dla zatwierdzania i realizacji wyciągu płatności (Approve Payment).

FB01 + PAY01:

FB01 to bezpośredni wpis księgowy w SAP (Post Financial Document).

Połączyliśmy go z transakcją płatniczą PAY01.

ME21N + MIRO:

ME21N to flagowa transakcja SAP MM do tworzenia zamówień zakupu (Create Purchase Order).

MIRO to oficjalna transakcja SAP do wprowadzania i weryfikacji faktur zakupu (Enter Incoming Invoice).

user role dataet 
tutaj robimy sztuczny rejestr uzytkownikow i przypisanych im transakcji 
 
 tworzymy liste slownikow ktora symuluje pobrane z systemu sap uprawneiania i rolekonkretnych uzytwkonikow 
kazdy slownik reprezentuje jeden wpis w rejestrze i zawiera id, kod transakcji w sao do ktorej dany uzytkownaik ma dostep i nazwa roli biznesoej


potem towrzyny zestawienia naruszen 

df_user_roles = pd.DataFrame(user_roles_data): Przekształca listę słowników w dwuwymiarową ramkę danych Pandas (DataFrame). Klucze słowników stają się nagłówkami kolumn.

ogolnie piszemy najpier wslowniki bo jest to bardziej optymalne 

In [ ]:
Wyrkywanie naruszen sod proprzez dopasowanie par do macierzy ryzyk 

to wazne git checkout -- ../.gitignore

co zmienilam w kodzie?

1 Najpierw ustawiamy ignorowanie folderu output/, żeby wygenerowane pliki Excel nie śmieciły w repozytorium.
2 Teraz zabezpieczymy wczytywanie plików CSV (wielkość liter upper(), usuwanie spacji strip(), wczytywanie kodów jako stringi dtype=str oraz sprawdzanie nagłówków).
3 Na koniec zmieniamy samą logikę złączenia z heavy self-join na bezpośrednie łączenie macierzy z rolami oraz dodajemy arkusz podsumowania w Excelu.

Nazwy zmiennych głównych ramek (DataFrame):

W notebooku używaliśmy nazw df_user_roles i df_sod.

W funkcji main() skryptu main.py nazwałam je skrótowo df_roles i df_sod, ale przekazują dokładnie te same dane.

Nazwy funkcji:

W notebooku wykonywaliśmy operacje bezpośrednio w komórkach (np. pd.read_csv(...) czy pd.merge(...)).

W main.py opakowaliśmy ten sam kod w czyste funkcje: load_sod_matrix(), load_user_roles() oraz detect_sod_conflicts(), co jest standardem produkcyjnym.

Nazwy kolumn:

Wszystkie kolumny (user_id, tcode, tcode_a, tcode_b, risk_id, risk_level, description) są w 100% identyczne w obu plikach.

Każda funkcja w pliku tests/test_sod.py działa według klasycznego schematu testów jednostkowych AAA (Arrange, Act, Assert):

Arrange (Przygotuj) – Tworzy w pamięci sztuczne, małe tabele DataFrame (udające pliki CSV).

Act (Działaj) – Uruchamia Twoją funkcję z main.py na tych próbnych danych.

Assert (Sprawdź) – Weryfikuje, czy wynik zgadza się z oczekiwaniami (jeśli nie, pytest krzyczy na czerwono).

1. test_detect_sod_conflicts_basic() – Test „ścieżki optymistycznej” (Happy Path)
Co przygotowuje (Arrange):

Tworzy użytkownika USR01, który posiada dokładnie dwie transakcje: FB60 (księgowanie faktury) i F-53 (wykonanie płatności).

Tworzy macierz ryzyka z jedną zasadą (RSK01), mówiącą, że połączenie FB60 i F-53 to konflikt CRITICAL.

Co uruchamia (Act): Wywołuje detect_sod_conflicts(user_roles, sod_matrix).

Co sprawdza (Assert):

assert len(conflicts) == 1 – upewnia się, że skrypt znalazł dokładnie jeden konflikt (ani zera, ani duplikatów).

assert conflicts.iloc[0]["user_id"] == "USR01" – sprawdza, czy konflikt został przypisany właściwemu użytkownikowi.

assert conflicts.iloc[0]["risk_id"] == "RSK01" – sprawdza, czy poprawnie zidentyfikowano numer ryzyka.

2. test_detect_sod_conflicts_case_insensitivity() – Test odporności na wielkość liter
Co przygotowuje (Arrange):

Tworzy użytkownika USR02, który ma znormalizowane kody transakcji FB60 oraz F-53 w swoich rolach, oraz standardową regułę w macierzy ryzyka.

(Wskazówka: Jeśli chcesz przetestować same małe litery fb60, trzeba by przekazać je przez load_user_roles lub zadbać o normalizację w samej funkcji detect_sod_conflicts).

Co uruchamia (Act): Wywołuje detect_sod_conflicts(user_roles, sod_matrix).

Co sprawdza (Assert):

assert len(conflicts) == 1 – potwierdza, że mechanizm łączenia (merge) zadziałał poprawnie i nie pominął rekordu.

3. test_no_conflicts_when_roles_do_not_overlap() – Test braku fałszywych alarmów (False Positives)
Co przygotowuje (Arrange):

Tworzy użytkownika USR03, który posiada FB60 (księgowość) oraz VA01 (tworzenie zamówienia sprzedaży).

Macierz ryzyka nadal szuka konfliktu dla pary FB60 + F-53.

Co uruchamia (Act): Wywołuje detect_sod_conflicts(user_roles, sod_matrix).

Co sprawdza (Assert):

assert conflicts.empty – sprawdza, czy zwrócona tabela jest całkowicie pusta. Dzięki temu masz pewność, że wewnętrzne złączenie (inner join) w pandas nie generuje fałszywych alarmów, gdy użytkownik ma tylko połowę toksycznej pary.

4. test_generate_summary_structure() – Test generatora podsumowania
Co przygotowuje (Arrange):

Tworzy gotową tabelę z jednym wykrytym konfliktem na poziomie HIGH.

Co uruchamia (Act): Wywołuje funkcję generate_summary(conflicts).

Co sprawdza (Assert):

assert not summary.empty – upewnia się, że tabela podsumowująca nie jest pusta.

assert "Total Conflicts" in summary["Risk Level / Metric"].values – sprawdza, czy w tabeli znajduje się wymagany wiersz sumujący (Total Conflicts), który potem trafia do pierwszego arkusza w Excelu.

# wyjaśnienie testów
czesc 1 to happy path 
Wszystkie te testy napisałem na podstawie tego, jak zbudowaliśmy funkcje detect_sod_conflicts oraz generate_summary w pliku main.py.

Twój algorytm w main.py robi dwie kluczowe rzeczy:

Lączy (merge) uprawnienia użytkownika z macierzą ryzyka po kodach T-Code.

Podsumowuje wyniki (groupby), zliczając konflikty według poziomów ryzyka (CRITICAL, HIGH itd.).

Dlatego testujemy 4 konkretne sytuacje, żeby mieć 100% pewności, że kod nie wyłoży się na produkcji.


Część 2: Brak nakładających się ról (brak fałszywych alarmów) negative path 
Logika i mechanizm
test_no_conflicts_when_roles_do_not_overlap(): Tworzymy użytkownika USR03, który posiada transakcję FB60 (księgowość) oraz VA01 (tworzenie zamówienia sprzedaży). Reguła w macierzy ryzyka szuka jednak pary FB60 + F-53.

Dlaczego to testujemy: Upewniamy się, że nasz podwójny merge (inner join) w pandas nie wygeneruje błędu ani pustego rekordu-widma, gdy użytkownik ma tylko połowę toksycznej pary. Zwrócona tabela conflicts musi być całkowicie pusta (conflicts.empty).
Oto 3 konkretne powody, dlaczego ten test jest kluczowy:

Testowanie „połówki” konfliktu:
Użytkownik w naszym teście ma transakcję FB60 (pierwszy element toksycznej pary FB60 + F-53), ale zamiast płatności ma VA01 (sprzedaż). W pandas bardzo łatwo przypadkiem zmienić how="inner" na how="left" podczas pisania merge. Gdybyś to zrobiła, skrypt zgłosiłby konflikt dla każdego, kto ma tylko FB60 — zalewając audytora tysiącami fałszywych zgłoszeń.

Ochrona przed przyszłymi zmianami (regresja):
Gdy za miesiąc wrócisz do kodu i zechcesz go ulepszyć (np. dodać obsługę innych kolumn), ten test natychmiast zapali się na czerwono, jeśli przypadkiem zepsujesz logikę filtrowania i skrypt zacznie wypluwać konflikty tam, gdzie ich nie ma.

Sygnał dla rekrutera:
Każdy junior testuje tylko tzw. Happy Path (czy funkcja działa, gdy dane są poprawne i konflikt istnieje). Dodanie testu negatywnego (Negative Test Case) pokazuje, że myślisz jak inżynier danych i audytor — sprawdzasz nie tylko czy silnik łapie błędy, ale czy nie oskarża niewinnych użytkowników.

pytest korzysta z wbudowanego mechanizmu tzw. Test Discovery (automatycznego wykrywania testów), który opiera się na 3 sztywnych konwencjach nazewnictwa:

Nazwy plików: Gdy wpisujesz py -m pytest w katalogu głównym (~/Desktop/SAP), narzędzie rekurencyjnie skanuje wszystkie podfoldery w poszukiwaniu plików pasujących do wzorca test_*.py lub *_test.py. Dlatego od razu znalazło tests\test_sod.py, a zignorowało np. main.py.   
image_60a87a.png
+ 2

Nazwy funkcji: Wewnątrz znalezionych plików pytest szuka wyłącznie funkcji, których nazwa zaczyna się od przedrostka test_ (np. def test_no_conflicts_when_roles_do_not_overlap():). Gdybyś nazwała funkcję po prostu check_conflicts(), pytest by ją pominął (traktując ją jako zwykłą funkcję pomocniczą).   
image_60a87a.png

Klasy (opcjonalnie): Jeśli ktoś grupuje testy w klasy, muszą one zaczynać się od Test* (bez metody __init__).

Dzięki tym trzem zasadom nie musisz pisać żadnego pliku konfiguracyjnego ani ręcznie podpinać testów do main.py — wystarczy trzymać się nazewnictwa test_..., a pytest sam wszystko odnajdzie i odpali.   
image_60a87a.png
+ 1